In [5]:
# Imports
import sys
from pathlib import Path
import warnings

ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models.pategan.models import PATEGAN

warnings.filterwarnings("ignore", message="X does not have valid feature names")

# ---------- Custom PATEGAN for CAR ----------
class PATEGAN(PATEGAN):
    def __init__(self):
        super().__init__(
            epsilon=5.0,     # relaxed privacy → better scores
            delta=1e-5,
            num_teachers=5,  # fewer teachers → more data per teacher
            niter=8000,      # you can change to 1000/8000 later
            batch_size=64,
            learning_rate=5e-4,
            lambda_gp=5.0,
            random_state=42,
        )

# ---------------- Preprocess data (CAR) ----------------
dataset_path = ROOT / "raw_data" / "adult.csv"
output_path = ROOT / "discretized_data" / "adult.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

discretize_preprocess(str(dataset_path), str(output_path))

# ---------------- Run Train/Test/Synthetic pipeline ----------------
input_csv = str(output_path)          # discretised CAR data
output_dir = str(ROOT / "sample_data" / "adult")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "adult" / "pategan")

model_preview = PATEGAN()
print("PATEGAN will train for:", model_preview.niter, "iterations")

pipeline = TrainTestSplitPipeline(model=PATEGAN)

result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
)

print(result)


ROOT set to: C:\Users\Prabu\Downloads\Katabatic
Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\adult.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\adult.csv
PATEGAN will train for: 8000 iterations
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
Remapped y classes: [0, 1] -> [0, 1]
Loaded training data: X shape=(26048, 14), y shape=(26048,)
Training PATE-GAN with ε=5.0, δ=1e-05
Teachers: 5, Iterations: 8000


Training: 100%|██████████| 8000/8000 [03:23<00:00, 39.27it/s]


Training completed!

Generating 26048 synthetic samples...


C:\Users\Prabu\Downloads\Katabatic\katabatic\models\pategan\models.py:503: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_synth[y_col] = y_synth[y_col].astype(int)


Remapped y_test.csv to match synthetic data encoding
Saved x_synth.csv to C:\Users\Prabu\Downloads\Katabatic\synthetic\adult\pategan\x_synth.csv
Saved y_synth.csv to C:\Users\Prabu\Downloads\Katabatic\synthetic\adult\pategan\y_synth.csv
Saved metadata.json to C:\Users\Prabu\Downloads\Katabatic\synthetic\adult\pategan\metadata.json

Results saved to: Results\adult\pategan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6955
F1 Score: 0.6881
AUC: 0.6750

MLP:
Accuracy: 0.6174
F1 Score: 0.6419
AUC: 0.6051

RF:
Accuracy: 0.2919
F1 Score: 0.2063
AUC: 0.4480

XGBoost:
Accuracy: 0.7530
F1 Score: 0.6659
AUC: 0.5291
Train test split pipeline executed successfully.


In [6]:
import pandas as pd

rows = [
    {"Model": "LR",      "Metric": "Accuracy", "Value": 0.6955},
    {"Model": "LR",      "Metric": "F1 Score", "Value": 0.6881},
    {"Model": "LR",      "Metric": "AUC",      "Value": 0.6750},

    {"Model": "MLP",     "Metric": "Accuracy", "Value": 0.6174},
    {"Model": "MLP",     "Metric": "F1 Score", "Value": 0.6419},
    {"Model": "MLP",     "Metric": "AUC",      "Value": 0.6051},

    {"Model": "RF",      "Metric": "Accuracy", "Value": 0.2919},
    {"Model": "RF",      "Metric": "F1 Score", "Value": 0.2063},
    {"Model": "RF",      "Metric": "AUC",      "Value": 0.4480},

    {"Model": "XGBoost", "Metric": "Accuracy", "Value": 0.7530},
    {"Model": "XGBoost", "Metric": "F1 Score", "Value": 0.6659},
    {"Model": "XGBoost", "Metric": "AUC",      "Value": 0.5291},
]

df = pd.DataFrame(rows)

save_path = r"C:\Users\Prabu\Downloads\petgan_adult_tstr.csv"
df.to_csv(save_path, index=False)

print("Saved CSV to:", save_path)
df


Saved CSV to: C:\Users\Prabu\Downloads\petgan_adult_tstr.csv


,Model,Metric,Value
0,LR,Accuracy,0.6955
1,LR,F1 Score,0.6881
2,LR,AUC,0.6750
3,MLP,Accuracy,0.6174
4,MLP,F1 Score,0.6419
5,MLP,AUC,0.6051
6,RF,Accuracy,0.2919
7,RF,F1 Score,0.2063
8,RF,AUC,0.4480
9,XGBoost,Accuracy,0.7530
